# C3 exp11 on v3: each segment's own examples, chosen by similarity (KATE)

Qwen2.5-7B-Instruct (4-bit), k = 0, definitions on, 2 examples per label, **data v3**. exp08 showed every segment the same 14 randomly drawn train examples (0.341); exp09 found that the draw alone moves the score by ~0.05. Here each segment gets **its own** 2 examples of every label, the ones closest to it (KATE: Liu et al. 2021, "What Makes Good In-Context Examples for GPT-3?"). The prompt is otherwise exp08's: same template, options, label order (cs first, other last).

Three runs, each with its own shots file (`scripts/retrieve_shots.py`; examples from train, one per label and transcript):
- **knn_tuned**: closest in the space of C1's encoder, DeBERTa-v3-base fine-tuned on train with C1's code and adopted setting (lr 1e-5, 3 epochs, seed 42, last epoch kept; `[CLS]` vector). The paper's best classifier-side variant (an encoder fine-tuned on the task itself).
- **knn_base**: closest in the space of DeBERTa-v3-base as released, not fine-tuned (mean of the last layer). Similarity of wording only.
- **random**: 2 random examples of every label, drawn for each segment (seed 0). The control: different examples per segment, without similarity.

Also written, without the 7B: `shots/<name>/nearest/` (each segment labelled with its closest example's label: the retriever alone) and `shots/knn_tuned/encoder/` (the fine-tuned encoder's own predictions).

Speed: `--prefix-cache`, batch 16. Only the instruction and options are shared now (the examples differ per segment), so the cache saves less than in exp08. If a run stops at its cache check, it runs again without the cache as `<run>_nocache`.

Note: `tasks/C3-prompting-baseline/experiments/exp11_knn-shots-7b/NOTE.md`. Temporary trials (`scripts/trial_prompting.py`); not reportable (PROTOCOL §4.1).

Before running, in the notebook settings: **GPU T4 x2**, **Internet on**, the `smi-v3-meetings` dataset attached (only that one: the data cell looks for v3).

Expected time: about 4 hours (an estimate; the first progress lines of each log show the real speed).

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Code and packages
`bitsandbytes` for the 4-bit 7B, `sentencepiece` for DeBERTa's tokenizer (as in C1's notebooks). Do **not** install `requirements.txt` here.

In [ ]:
!rm -rf /kaggle/working/repo
!git clone -b C3/prompting-baseline https://github.com/ahmedwaleedaref/Meeting-Intelligent-System-.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -3
!pip install -q transformers==5.20.1 bitsandbytes sentencepiece==0.2.2

## 3. Data folder: the v3 one

In [ ]:
import glob, json, os, subprocess, time

hits = [h for h in glob.glob("/kaggle/input/**/source.json", recursive=True) if json.load(open(h))["data_version"] == "v3"]
assert len(hits) == 1, f"expected exactly one v3 dataset under /kaggle/input, found {hits} : is smi-v3-meetings attached?"
DATA_ROOT = os.path.dirname(hits[0])
print("DATA_ROOT =", DATA_ROOT, "|", json.load(open(hits[0])))
os.makedirs("/kaggle/working/logs", exist_ok=True)

## 4. Download the models once

In [ ]:
from huggingface_hub import snapshot_download

SEVEN_B = "Qwen/Qwen2.5-7B-Instruct"
ENCODER = "microsoft/deberta-v3-base"
for name in (SEVEN_B, ENCODER):
    print(snapshot_download(name))

## 5. Helpers: shots files, runs, a queue per GPU, and waiting while showing progress
Every step logs to `logs/`. A shots file that already exists is not made again; a half-written one (folder without `shots.jsonl`) must be deleted by hand first.

In [ ]:
SHOTS = "/kaggle/working/shots"
RETRIEVE = {
    "knn_tuned": f"--select similar --encoder {ENCODER} --fine-tune --seed 42 --pooling cls",
    "knn_base": f"--select similar --encoder {ENCODER} --pooling mean",
    "random": "--select random --seed 0",
}

def retrieve(name):
    """make the shots file SHOTS/<name> , unless it is already there"""
    return (f"[ -f {SHOTS}/{name}/shots.jsonl ] || python scripts/retrieve_shots.py --data-root '{DATA_ROOT}' "
            f"--out {SHOTS}/{name} --per-class 2 {RETRIEVE[name]} > /kaggle/working/logs/shots_{name}.log 2>&1")

def score(name):
    """the 7B with each segment's examples from SHOTS/<name> ; if the prefix-cache check stops it , again without the cache"""
    run = f"c3_exp11_7b_{name}"
    command = (f"python scripts/trial_prompting.py --data-root '{DATA_ROOT}' --out-dir /kaggle/working/runs "
               f"--model {SEVEN_B} --load-in-4bit --shots-file {SHOTS}/{name} --batch-size 16")
    return (f"( {command} --run-name {run} --prefix-cache > /kaggle/working/logs/{run}.log 2>&1 || "
            f"{command} --run-name {run}_nocache > /kaggle/working/logs/{run}_nocache.log 2>&1 )")

def queue(gpu, commands):
    """run the commands one after another on one GPU , in the background"""
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONPATH="src", TOKENIZERS_PARALLELISM="false")
    # "; " : a failed step does not stop the next one in the queue
    return subprocess.Popen("; ".join(commands), shell=True, cwd="/kaggle/working/repo", env=env)

def last_line(path):
    lines = [l for l in open(path).read().splitlines() if l.strip()]
    return lines[-1] if lines else "(starting)"

def wait(processes):
    """every 2 minutes , print the latest line of every log , until all queues have finished"""
    while any(p.poll() is None for p in processes):
        time.sleep(120)
        for path in sorted(glob.glob("/kaggle/working/logs/*.log")):
            print(f"{os.path.basename(path)[:-4]:30} {last_line(path)[-80:]}")
        print()
    print("all queues finished")

## 6. Run both GPUs (~4 h)
- GPU 0: fine-tune the encoder and make `knn_tuned` (~25 min), then its 7B run.
- GPU 1: `random` (1 min, no GPU needed) and its 7B run, then `knn_base` (~5 min) and its 7B run.

If the session stops, run this cell again: finished steps are skipped and an interrupted 7B run continues where it stopped.

In [ ]:
wait([
    queue(0, [retrieve("knn_tuned"), score("knn_tuned")]),
    queue(1, [retrieve("random"), score("random"), retrieve("knn_base"), score("knn_base")]),
])

## 7. Results
Baselines (v3): exp08's one fixed draw of 2 examples per label, 0.341; zero-shot 0.206. Prompt noise (exp07): differences below ~0.035 are not real. C1's encoder (v4, k = 0, 3 seeds): 0.5225.

In [ ]:
def show(name, path):
    if not os.path.exists(f"{path}/metrics.json"):
        print(f"{name:32} (missing : see logs/)"); return
    m = json.load(open(f"{path}/metrics.json"))
    predicted_other = sum(json.loads(line)["label"] == "other" for line in open(f"{path}/predictions.jsonl"))
    per_class = " ".join(f"{label} {m['per_class'][label]['f1']:.3f}" for label in ("cs", "co", "aa", "bk", "ar", "cc"))
    print(f"{name:32} {m['primary']['macro_f1_6']:8.4f} {predicted_other:11,}   {per_class}")

print(f"{'':32} {'macro-F1':>8} {'pred. other':>11}   (gold other 13,634)")
for name in RETRIEVE:
    run = f"c3_exp11_7b_{name}"
    if not os.path.exists(f"/kaggle/working/runs/{run}/metrics.json"):
        run += "_nocache"
    show(run, f"/kaggle/working/runs/{run}")
for name in ("knn_tuned", "knn_base"):
    show(f"{name} : retriever alone", f"{SHOTS}/{name}/nearest")
show("knn_tuned : encoder alone", f"{SHOTS}/knn_tuned/encoder")

## 8. Download the outputs
Right-hand panel: **Output** → `/kaggle/working` → `c3_exp11_runs.zip` → **⋮** → **Download**.

In [ ]:
%cd /kaggle/working
!zip -qr c3_exp11_runs.zip runs shots logs
!ls -la c3_exp11_runs.zip